# 第 6 章：让检索知识走向可执行查询

检索增强生成（RAG, Retrieval-Augmented Generation）为模型提供外部证据；自然语言转结构化查询语言（NL2SQL, Natural Language to SQL）还要把业务问题变成可执行的结构化查询语言（SQL, Structured Query Language）。本章以项目里的电商分析助手为例，串起“检索口径 → 生成查询 → 安全执行 → 解释结果”。智能体（Agent）由智能体开发工具包（ADK, Agent Development Kit）管理，大语言模型（LLM, Large Language Model）通过工具接触外部世界。检索结果的稳定文档标识符（ID, Identifier）让业务依据可以追踪。

**学习目标**：能说明三种工具各自的职责；读懂净销售额的口径和 SQL；分辨提示词约束与代码/数据库强制约束；离线验证危险 SQL 被拒绝，并按需检查真实数据库。**运行约定**：从项目根目录启动已安装依赖的 Jupyter 内核。默认所有代码只读本地文件或执行纯函数，不连接数据库，不请求云模型。首次使用请在项目根目录运行 uv sync --extra dev，并用项目虚拟环境启动 Jupyter。


## 6.1 场景：同一句“净销售额”隐藏了三类问题

用户问：“2026 年 8 月华东地区的净销售额是多少？”模型必须先找到本公司的“净销售额”定义，确认地区字段和表连接，再执行数据库查询。若直接凭语言常识回答，它可能用下单时间代替支付时间、把所有退款都扣掉，或者在连接多笔退款时重复累计支付额。

~~~mermaid
flowchart LR
  Q[用户问题] --> A[ADK 智能体]
  A --> K[search_knowledge<br/>找指标与连接口径]
  K --> R[(rag.documents)]
  K --> A
  A --> I[inspect_schema<br/>必要时核对真实字段]
  I --> D[(analytics 业务表)]
  I --> A
  A --> X[execute_sql<br/>执行受控 SELECT]
  X --> V[SQLGlot 校验]
  V --> RO[只读角色与事务]
  RO --> D
  D --> A
  A --> O[结论、口径、SQL 与知识文档 ID]
~~~

**证据分工**：<code>db/knowledge.json</code> 中的 <code>metric.net_sales</code> 解释业务定义；<code>analytics</code> 表提供实际金额；<code>execute_sql</code> 返回的结果才是数值证据。检索文档里的指令属于低信任数据，不能覆盖智能体和工具的规则。


In [1]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "tutorial":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "请从项目根目录或 tutorial 目录启动 Notebook"

# 只有读者主动改为 True，后续对应代码才会连接数据库或请求在线向量模型。
RUN_DB_READ = False
RUN_ONLINE_RETRIEVAL = False
print("项目根目录:", ROOT)
print("数据库只读示例:", "启用" if RUN_DB_READ else "跳过")
print("在线检索示例:", "启用" if RUN_ONLINE_RETRIEVAL else "跳过")


项目根目录: D:\Projects\PythonProject\nl2sql-rag
数据库只读示例: 跳过
在线检索示例: 跳过


## 6.2 项目代码中的工程链

| 步骤 | 当前实现 | 为什么需要 |
|---|---|---|
| 知识准备 | <code>db/knowledge.json</code> → <code>nl2sql.knowledge.load_knowledge()</code> | 把审核过的业务卡片与数据库当前字段组合，避免只记住过期列名 |
| 索引 | <code>nl2sql.indexer.index_knowledge()</code> | 文档用 Qwen 的 <code>text_type="document"</code> 向量化，按内容哈希增量更新 |
| 召回 | <code>nl2sql.retrieval.retrieve(question, limit=7)</code> | 查询用 <code>text_type="query"</code> 向量化；余弦排序与显式别名召回经倒数排名融合（RRF, Reciprocal Rank Fusion）合并，再补表依赖 |
| 决策 | <code>nl2sql_agent.agent.root_agent</code> | DeepSeek 在 ADK 中决定是否检索、核验字段、构造 SQL 或澄清 |
| 执行 | <code>nl2sql.sql_runner.safe_run_query(sql)</code> | 对模型输出做语法、权限和资源约束，并把错误变成工具可读的结果 |

仓库中 <code>root_agent</code> 的指令要求每个新主题先检索，必要时查结构，除澄清问题外必须执行 SQL。这里的“要求”是行为引导；实际权限边界还要靠工具和数据库实现。<code>search_knowledge</code> 返回稳定文档 ID，方便在回答和评估中追踪依据。

以下只导入工具并查看签名；导入模块不会向 DeepSeek 或 Qwen 发请求。


In [2]:
from inspect import signature
from nl2sql_agent.agent import root_agent, search_knowledge, inspect_schema, execute_sql

print("智能体名称:", root_agent.name)
for tool in (search_knowledge, inspect_schema, execute_sql):
    print(tool.__name__, signature(tool))
assert root_agent.name == "nl2sql_agent"


智能体名称: nl2sql_agent
search_knowledge (question: 'str') -> 'dict'
inspect_schema (table_names: 'list[str]') -> 'dict'
execute_sql (sql: 'str') -> 'dict'


## 6.3 为什么让 Agent 选工具，而不写死两步流程？

| 方案 | 适用情况 | 优点 | 代价与失败方式 |
|---|---|---|---|
| 固定流程：检索 → 填上下文 → 生成 | 一类简单问答、工具少 | 调用轨迹稳定，便于测试和控制费用 | 多轮追问或多表分析需要在代码里增加分支 |
| Agent 决策式 RAG | 需检索、查结构、执行查询并可能澄清 | 可根据问题选择下一步、处理一次执行错误 | 工具路径和成本更难预测，必须限制工具权限并评估轨迹 |
| 直接让模型生成 SQL | 固定 schema 且口径简单的内部原型 | 链路短 | 新口径、字段变更、权限与真实数值缺少独立核对 |

本项目选择第二种，但工具不是无限授权：<code>inspect_schema</code> 只接受六张业务表；<code>execute_sql</code> 只接受受控的读取查询。若后续接入文档搜索、应用程序编程接口（API, Application Programming Interface）或更多数据库，应逐一明确输入、可见范围、超时与审计字段。


## 6.4 从知识卡推导 SQL，而不是猜指标

净销售额的知识卡指定：先选时间窗口内的成功支付订单，扣除这些订单全部已批准退款；时间窗口按 <code>payments.paid_at</code>，退款应先按 <code>order_id</code> 聚合。地区来自 <code>customers.region</code>，通过 <code>orders.customer_id</code> 连接。以下只读取本地 JSON，不需要数据库或模型密钥。


In [3]:
import json

cards = {item["id"]: item for item in json.loads((ROOT / "db/knowledge.json").read_text(encoding="utf-8"))}
for doc_id in ("metric.net_sales", "join.customer_orders", "table.payments", "table.refunds"):
    item = cards[doc_id]
    print(f"[{doc_id}] {item['content']}")
assert {"payments", "refunds", "orders"} <= set(cards["metric.net_sales"]["related_tables"])


[metric.net_sales] 净销售额：在选定的成功支付订单集合中，SUM(payments.amount) 减去这些订单的 APPROVED 退款金额。先按 order_id 聚合退款再 LEFT JOIN，避免多笔退款放大支付金额。时间窗口默认以 payments.paid_at 选择订单；即使退款发生在窗口之后，也从所选订单金额中扣除。
[join.customer_orders] 地区和订单连接：analytics.orders.customer_id = analytics.customers.customer_id。地区筛选用 customers.region，城市筛选用 customers.city。
[table.payments] 支付表 analytics.payments：一张订单最多一笔支付；order_id 唯一，amount 为成功支付金额，paid_at 为支付时间，status='SUCCEEDED' 表示成功。按支付日期统计收入应使用 paid_at。
[table.refunds] 退款表 analytics.refunds：一张订单可以有多笔退款；只有 status='APPROVED' 的 amount 才计入退款金额。关联订单时应先按 order_id 聚合退款，避免把一笔支付重复计算。


**连接放大的反例**：一笔支付 100 元，关联两笔退款 10 元和 20 元，直接连接后支付行变成两行，求和会得到 170 元；先按订单聚合退款，正确结果为 70 元。种子数据目前每单最多产生一笔退款，但 <code>db/schema.sql</code> 允许多笔；工程设计必须遵守数据约束，而不能依赖当前样本“碰巧没出错”。


In [4]:
from decimal import Decimal

payment = Decimal("100.00")
refunds = [Decimal("10.00"), Decimal("20.00")]
wrong = sum((payment - refund for refund in refunds), Decimal("0.00"))
right = payment - sum(refunds, Decimal("0.00"))
print({"直接连接后求和": wrong, "先按订单汇总退款": right})
assert wrong == Decimal("170.00") and right == Decimal("70.00")


{'直接连接后求和': Decimal('170.00'), '先按订单汇总退款': Decimal('70.00')}


## 6.5 四道 SQL 边界

抽象语法树（AST, Abstract Syntax Tree）校验由 <code>nl2sql.sql_runner.validate_sql</code> 执行：SQLGlot 按 PostgreSQL 方言解析，只接受一条顶层 <code>SELECT</code>，检查整棵语法树中的写入节点、锁、表白名单和危险函数。通用表表达式（CTE, Common Table Expression）里的物理表也会检查。合法语句随后由数据库只读角色 <code>nl2sql_reader</code> 执行，事务设置 <code>READ ONLY</code>、5 秒语句超时与固定搜索路径；最外层限制返回 100 行。

> 100 行限制约束返回量，不能保证数据库只扫描 100 行。函数黑名单也无法证明任意 SELECT 都无副作用。生产环境还需身份授权、租户隔离、敏感字段控制、查询成本治理与审计。

下面只调用纯校验函数。对明显危险的请求演示 <code>safe_run_query</code> 的错误格式：校验会在建立数据库连接前拒绝它。


In [5]:
from nl2sql.sql_runner import validate_sql, safe_run_query

cases = {
    "合法读取": "SELECT COUNT(*) AS n FROM analytics.orders",
    "合法 CTE": "WITH p AS (SELECT order_id FROM analytics.payments) SELECT COUNT(*) FROM p",
    "字符串不是命令": "SELECT 'DELETE' AS literal_text",
    "直接写入": "DELETE FROM analytics.orders",
    "多语句": "SELECT 1; DROP TABLE analytics.orders",
    "越界模式": "SELECT * FROM rag.documents",
    "危险函数": "SELECT pg_sleep(1)",
}
for label, statement in cases.items():
    try:
        validate_sql(statement)
        verdict = "允许"
    except ValueError as exc:
        verdict = f"拒绝：{exc}"
    print(f"{label:10} {verdict}")

assert safe_run_query("DELETE FROM analytics.orders")["ok"] is False


合法读取       允许
合法 CTE     允许
字符串不是命令    允许
直接写入       拒绝：Only one SELECT statement is allowed
多语句        拒绝：Only one SELECT statement is allowed
越界模式       拒绝：Only analytics schema is allowed
危险函数       拒绝：Function is not allowed


把第 6.4 节口径落实到 SQL：先在 CTE 中按订单汇总已批准退款，再限定成功支付、华东地区和支付时间半开区间 <code>[2026-08-01, 2026-09-01)</code>。下面仅验证 SQL 能通过项目的解析器，默认不执行。


In [6]:
NET_SALES_SQL = """
WITH refunds_by_order AS (
  SELECT order_id, SUM(amount) AS refunded
  FROM analytics.refunds
  WHERE status = 'APPROVED'
  GROUP BY order_id
)
SELECT COALESCE(SUM(p.amount - COALESCE(r.refunded, 0)), 0) AS net_sales
FROM analytics.payments AS p
JOIN analytics.orders AS o ON o.order_id = p.order_id
JOIN analytics.customers AS c ON c.customer_id = o.customer_id
LEFT JOIN refunds_by_order AS r ON r.order_id = p.order_id
WHERE p.status = 'SUCCEEDED'
  AND c.region = '华东'
  AND p.paid_at >= '2026-08-01'
  AND p.paid_at < '2026-09-01'
"""
print("校验通过:", validate_sql(NET_SALES_SQL).splitlines()[-1])


校验通过:   AND p.paid_at < '2026-09-01'


### 可选的数据库与在线检索观察

如果已按项目根目录 <code>README.md</code> 完成数据库迁移、种子与索引，可自行把第一个开关改为 <code>True</code>：读取实际字段和净销售额。项目集成测试对固定样例数据的金标准为 <code>11261.00</code>。第二个开关还需 Qwen 密钥与有效索引；它调用在线向量模型并读取数据库。两项默认均跳过，代码不会修改业务数据或索引。


In [7]:
if RUN_DB_READ:
    schema_result = inspect_schema(["payments", "refunds", "orders", "customers"])
    print("实际字段数:", len(schema_result["columns"]))
    result = execute_sql(NET_SALES_SQL)
    print("查询结果:", result)
    assert result["ok"] and result["rows"][0]["net_sales"] == "11261.00"
else:
    print("已跳过数据库读取；需要时将 RUN_DB_READ 设为 True。")

if RUN_ONLINE_RETRIEVAL:
    assert RUN_DB_READ, "在线检索也依赖已初始化的数据库；先启用 RUN_DB_READ"
    knowledge = search_knowledge("2026 年 8 月华东地区的净销售额是多少？")
    print("命中文档 ID:", knowledge.get("doc_ids"))
    assert knowledge["ok"] and "metric.net_sales" in knowledge["doc_ids"]
else:
    print("已跳过 Qwen 在线检索；需要时将 RUN_ONLINE_RETRIEVAL 设为 True。")


已跳过数据库读取；需要时将 RUN_DB_READ 设为 True。
已跳过 Qwen 在线检索；需要时将 RUN_ONLINE_RETRIEVAL 设为 True。


## 6.6 如何验证整条链

<code>tests/unit/test_sql_runner.py</code> 检查危险 SQL 被拒绝和合法 CTE 通过；<code>tests/integration/test_database.py</code> 逐笔独立累计支付与退款，核对 8 月华东净销售额；<code>tests/eval/core.evalset.json</code> 关注“先检索、后执行”的工具轨迹，<code>tests/eval/safety.evalset.json</code> 检查写入请求不触发工具。轨迹正确不等于数值正确：评估需要同时看必要文档是否召回、SQL 是否符合口径、数据库结果与最终中文表述。

**基础练习**：用户问“2026 年 8 月华东有多少笔非取消订单？”从知识卡找出计数粒度、时间字段和地区连接，写一条只读 SQL 并先交给 <code>validate_sql</code>。**检查标准**：使用 <code>COUNT(DISTINCT o.order_id)</code> 或等价的订单粒度计数；筛选 <code>orders.ordered_at</code>、<code>customers.region</code>、<code>status &lt;&gt; 'CANCELLED'</code>；若启用数据库读取，结果应与 <code>tests/integration/test_database.py</code> 的华东 8 月基线 19 一致。

**进阶练习**：构造“CTE 中尝试写入”“跨模式读取”“锁行”“多语句”四个输入，记录 <code>validate_sql</code> 的拒绝原因。再设计一个新规则：当用户提出“按退款发生月统计退款”时，与现有“支付窗口内订单的净销售额”口径如何区分？**检查标准**：危险请求在连接数据库前被拒绝；能指出新增规则应放在知识卡、提示词还是工具边界，并说明若修改知识卡需重建索引。

**反思**：如果检索返回正确的 <code>metric.net_sales</code>，但生成 SQL 用 <code>refunds.refunded_at</code> 限定退款，你会优先检查哪一层？如果 SQL 正确却回答里把 8 月说成 9 月，又应检查哪一层？
